# 🌸 Iris Flower Species Classifier

### CodeAlpha Data Science Internship Project

**Objective:** Develop a Machine Learning classification model that predicts Iris flower species based on sepal and petal measurements.

**Models Used:**
- Decision Tree Classifier
- K-Nearest Neighbors (KNN)
- Logistic Regression

---

## Step 1: Import Libraries

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
import joblib

# Plot settings
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150
palette = ['#2ecc71', '#3498db', '#e74c3c']

print('Libraries imported successfully.')

## Step 2: Load Dataset

In [ ]:
# Load the Iris dataset from the data folder
df = pd.read_csv('../data/Iris.csv')

# Drop the Id column as it's not a feature
if 'Id' in df.columns:
    df = df.drop(columns=['Id'])

print(f'Dataset loaded successfully: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

## Step 3: Inspect Dataset

In [ ]:
# Shape of the dataset
print(f'Shape: {df.shape}')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')

In [ ]:
# Data types
df.dtypes

In [ ]:
# Dataset info
df.info()

## Step 4: Check Missing Values

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
print('Missing values per column:')
print(missing_values)
print(f'\nTotal missing values: {missing_values.sum()}')

## Step 5: Check Duplicate Values

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f'Number of duplicate rows: {duplicates}')

## Step 6: Summary Statistics

In [ ]:
# Descriptive statistics
df.describe()

In [ ]:
# Class distribution
print('Class distribution:')
print(df['Species'].value_counts())

## Step 7: Exploratory Data Analysis (EDA)

Visualize the relationships between features and species.

In [ ]:
# Pair Plot
g = sns.pairplot(df, hue='Species', palette=palette, height=2.5,
                 plot_kws={'alpha': 0.7, 'edgecolor': 'white', 'linewidth': 0.5})
g.figure.suptitle('Pairplot of Iris Features by Species', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap
feature_cols = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']
feature_labels = ['Sepal Length', 'Sepal Width', 'Petal Length', 'Petal Width']

fig, ax = plt.subplots(figsize=(8, 6))
corr = df[feature_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1,
            square=True, fmt='.2f', linewidths=1, ax=ax)
ax.set_title('Correlation Heatmap of Iris Features', fontsize=14, fontweight='bold', pad=15)
ax.set_xticklabels(feature_labels, rotation=45, ha='right')
ax.set_yticklabels(feature_labels, rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Histograms
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for i, (feature, label) in enumerate(zip(feature_cols, feature_labels)):
    sns.histplot(data=df, x=feature, hue='Species', palette=palette, kde=True, ax=axes[i], alpha=0.7)
    axes[i].set_title(f'Distribution of {label}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(f'{label} (cm)')
    axes[i].set_ylabel('Count')
fig.suptitle('Feature Distributions by Species', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Box Plots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for i, (feature, label) in enumerate(zip(feature_cols, feature_labels)):
    sns.boxplot(data=df, x='Species', y=feature, palette=palette, ax=axes[i], width=0.6)
    axes[i].set_title(f'{label} by Species', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Species')
    axes[i].set_ylabel(f'{label} (cm)')
fig.suptitle('Feature Box Plots by Species', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Step 8: Data Preprocessing

Split the dataset into training and test sets, and scale the features.

In [ ]:
# Define features and target
X = df[feature_cols]
y = df['Species']

# Split the dataset (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrames for readability
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_cols, index=X_test.index)

print(f'Training set: {X_train_scaled.shape[0]} samples')
print(f'Test set:     {X_test_scaled.shape[0]} samples')

## Step 9: Train Models

Train three classification models:
1. **Decision Tree** — splits data based on feature thresholds
2. **K-Nearest Neighbors** — classifies based on closest training examples
3. **Logistic Regression** — uses a linear decision boundary with softmax

In [ ]:
# Define models
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(max_iter=200, random_state=42)
}

# Train and evaluate each model
results = {}

for name, model in models.items():
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict
    predictions = model.predict(X_test_scaled)
    
    # Evaluate
    results[name] = {
        'model': model,
        'accuracy': accuracy_score(y_test, predictions),
        'precision': precision_score(y_test, predictions, average='weighted', zero_division=0),
        'recall': recall_score(y_test, predictions, average='weighted', zero_division=0),
        'f1_score': f1_score(y_test, predictions, average='weighted', zero_division=0),
        'predictions': predictions
    }
    
    print(f'{name}: Accuracy = {results[name]["accuracy"]:.4f}')

## Step 10: Compare Models

In [ ]:
# Create comparison DataFrame
comparison = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [r['accuracy'] for r in results.values()],
    'Precision': [r['precision'] for r in results.values()],
    'Recall': [r['recall'] for r in results.values()],
    'F1 Score': [r['f1_score'] for r in results.values()]
}).set_index('Model')

comparison.style.format('{:.4f}').highlight_max(axis=0, color='lightgreen')

In [ ]:
# Model comparison bar chart
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2ecc71', '#3498db', '#e74c3c']
model_names = list(results.keys())
accuracies = [results[n]['accuracy'] for n in model_names]

bars = ax.bar(model_names, accuracies, color=colors, width=0.5,
              edgecolor='white', linewidth=1.5)
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{acc:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## Step 11: Select Best Model

In [ ]:
# Select the model with highest accuracy
best_name = max(results, key=lambda k: results[k]['accuracy'])
best_model = results[best_name]['model']
best_accuracy = results[best_name]['accuracy']

print(f'Best Model: {best_name}')
print(f'Accuracy:   {best_accuracy:.4f}')
print(f'Precision:  {results[best_name]["precision"]:.4f}')
print(f'Recall:     {results[best_name]["recall"]:.4f}')
print(f'F1 Score:   {results[best_name]["f1_score"]:.4f}')

## Step 12: Evaluate Final Model

In [ ]:
# Classification Report
best_predictions = results[best_name]['predictions']
print(f'Classification Report for {best_name}:\n')
print(classification_report(y_test, best_predictions, zero_division=0))

In [ ]:
# Confusion Matrix
species_names = ['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']
cm = confusion_matrix(y_test, best_predictions)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=species_names, yticklabels=species_names,
            linewidths=1, linecolor='white', ax=ax)
ax.set_title(f'Confusion Matrix - {best_name}', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('True Label', fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

## Step 13: Save Model

In [ ]:
# Save the best model and scaler
save_dir = os.path.join('..', 'saved_model')
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, 'best_model.pkl')

save_data = {'model': best_model, 'scaler': scaler}
joblib.dump(save_data, save_path)

print(f'Model saved to: {save_path}')

## Step 14: Prediction Demo

In [ ]:
# Sample predictions
samples = [
    {'features': [5.1, 3.5, 1.4, 0.2], 'expected': 'Iris-setosa'},
    {'features': [6.7, 3.1, 4.7, 1.5], 'expected': 'Iris-versicolor'},
    {'features': [6.3, 2.7, 4.9, 1.8], 'expected': 'Iris-virginica'},
]

print(f'{"Input Features":<30} {"Expected":<18} {"Predicted":<18} {"Confidence"}')
print('-' * 85)

for sample in samples:
    features_array = np.array(sample['features']).reshape(1, -1)
    features_scaled = scaler.transform(features_array)
    
    prediction = best_model.predict(features_scaled)[0]
    probabilities = best_model.predict_proba(features_scaled)[0]
    confidence = max(probabilities)
    
    match = 'OK' if prediction == sample['expected'] else 'MISMATCH'
    print(f'{str(sample["features"]):<30} {sample["expected"]:<18} {prediction:<18} {confidence:.1%} {match}')

---

## Summary

- Loaded and explored the Iris dataset (150 samples, 4 features, 3 classes)
- Performed EDA with pair plots, heatmaps, histograms, and box plots
- Trained three models: Decision Tree, KNN, and Logistic Regression
- All models achieved high accuracy on this well-separated dataset
- The best model was saved for deployment in the Streamlit application

**Key Insight:** Petal measurements (length and width) are more discriminative than sepal measurements for distinguishing between species.

---
*CodeAlpha Data Science Internship Project*